In [1]:
import pkg_resources
pkg_resources.get_distribution("azureml_core").version

'1.0.55'

In [34]:
# Load Workspace information from config.json (it can be downloaded directly from Azure Portal)
from azureml.core import Workspace, Experiment
ws=Workspace.from_config("dagaldwestus.json")

print('Workspace name: ' + ws.name, 
      'Azure region: ' + ws.location, 
      'Subscription id: ' + ws.subscription_id, 
      'Resource group: ' + ws.resource_group, sep='\n')

Workspace name: dagaldwestus
Azure region: westus2
Subscription id: 5f08d643-1910-4a38-a7c7-84a39d4f42e0
Resource group: dagald


In [99]:
from sklearn.datasets import load_diabetes
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import train_test_split
from sklearn.externals import joblib

X, y = load_diabetes(return_X_y = True)
columns = ['age', 'gender', 'bmi', 'bp', 's1', 's2', 's3', 's4', 's5', 's6']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=0)
data = {
    "train":{"X": X_train, "y": y_train},        
    "test":{"X": X_test, "y": y_test}
}

print ("Data contains", len(data['train']['X']), "training samples and",len(data['test']['X']), "test samples")

Data contains 353 training samples and 89 test samples


In [100]:
# Get an experiment object from Azure Machine Learning
experiment = Experiment(workspace=ws, name="train-within-notebook")

# Create a run object in the experiment
run =  experiment.start_logging()
# Log the algorithm parameter alpha to the run
run.log('alpha', 0.03)

# Create, fit, and test the scikit-learn Ridge regression model
regression_model = Ridge(alpha=0.03)
regression_model.fit(data['train']['X'], data['train']['y'])
preds = regression_model.predict(data['test']['X'])

# Output the Mean Squared Error to the notebook and to the run
print('Mean Squared Error is', mean_squared_error(data['test']['y'], preds))
run.log('mse', mean_squared_error(data['test']['y'], preds))

# Save the model to the outputs directory for capture
model_file_name = 'outputs/model.pkl'

joblib.dump(value = regression_model, filename = model_file_name)

# upload the model file explicitly into artifacts 
run.upload_file(name = model_file_name, path_or_stream = model_file_name)

# Complete the run
run.complete()

WARNING - Converted key mse of value 3424.900315896018 to 3424.900315896018.



Mean Squared Error is 3424.900315896018


In [11]:
run

Experiment,Id,Type,Status,Details Page,Docs Page
train-within-notebook,2afad57f-1fc1-4c91-bb9e-612b64a41066,,NotStarted,Link to Azure Portal,Link to Documentation


In [101]:
import numpy as np
from tqdm import tqdm

# list of numbers from 0 to 1.0 with a 0.05 interval
alphas = np.arange(0.0, 1.0, 0.05)

# try a bunch of alpha values in a Linear Regression (Ridge) model
for alpha in tqdm(alphas):
    # create a bunch of runs, each train a model with a different alpha value
    with experiment.start_logging() as run:
        # Use Ridge algorithm to build a regression model
        regression_model = Ridge(alpha=alpha)
        regression_model.fit(X=data["train"]["X"], y=data["train"]["y"])
        preds = regression_model.predict(X=data["test"]["X"])
        mse = mean_squared_error(y_true=data["test"]["y"], y_pred=preds)

        # log alpha, mean_squared_error and feature names in run history
        run.log(name="alpha", value=alpha)
        run.log(name="mse", value=mse)

        # Save the model to the outputs directory for capture
        joblib.dump(value=regression_model, filename='outputs/model.pkl')



  0%|                                                                                           | 0/20 [00:00<?, ?it/s]WARNING - Converted key alpha of value 0.0 to 0.0.

WARNING - Converted key mse of value 3424.3166882137334 to 3424.3166882137334.

  5%|████▏                                                                              | 1/20 [00:05<01:50,  5.83s/it]WARNING - Converted key alpha of value 0.05 to 0.05.

WARNING - Converted key mse of value 3408.9153122589296 to 3408.9153122589296.

 10%|████████▎                                                                          | 2/20 [00:11<01:43,  5.76s/it]WARNING - Converted key alpha of value 0.1 to 0.1.

WARNING - Converted key mse of value 3372.6496278100326 to 3372.6496278100326.

 15%|████████████▍                                                                      | 3/20 [00:17<01:38,  5.78s/it]WARNING - Converted key alpha of value 0.15000000000000002 to 0.15000000000000002.

WARNING - Converted key mse of value 3345.

In [13]:
experiment

Name,Workspace,Report Page,Docs Page
train-within-notebook,dagaldwestus,Link to Azure Portal,Link to Documentation


In [102]:
runs = {}
run_metrics = {}

# Create dictionaries containing the runs and the metrics for all runs containing the 'mse' metric
for r in tqdm(experiment.get_runs()):
    metrics = r.get_metrics()
    if 'mse' in metrics.keys():
        runs[r.id] = r
        run_metrics[r.id] = metrics

# Find the run with the best (lowest) mean squared error and display the id and metrics
best_run_id = min(run_metrics, key = lambda k: run_metrics[k]['mse'])
best_run = runs[best_run_id]
print('Best run is:', best_run_id)
print('Metrics:', run_metrics[best_run_id])

# Tag the best run for identification later
best_run.tag("Best Run")

105it [00:14,  6.86it/s]


Best run is: ed951fad-40a4-45e2-8b16-8cedcc135f91
Metrics: {'alpha': 0.4, 'mse': 3295.741064355809}


In [104]:
# View the files in the run
for f in best_run.get_file_names():
    print(f)
    
# Register the model with the workspace
tags = {'tag1': "value of tag1", 'model': 'best model', 'testing_model': 'testing tags'}
model = best_run.register_model(model_name='tagged_model', model_path='outputs/model.pkl', tags=tags)

logs/user_log.txt
outputs/model.pkl


In [35]:
# score.py and myenv.yml should be in the same directoy as the notebook
from azureml.core.image import ContainerImage
image_config = ContainerImage.image_configuration(execution_script="score.py", 
                                    runtime="python", 
                                    conda_file="myenv.yml")


In [36]:
#from azureml.core.model import Model
#model = Model(ws, 'best_model', version=4)

In [37]:
model.version

4

In [38]:
image = ContainerImage.create(name = "azmlmona16dev3",
                              # this is the model object
                              models = [model],
                              image_config = image_config,
                              workspace = ws)

image.wait_for_creation(show_output = True)

Creating image
Running.........................................................
Succeeded
Image creation operation finished for image azmlmona16dev3:3, operation "Succeeded"


In [14]:
#image = ContainerImage(ws, "azmlmona16dev3")

In [39]:
# Enable data collector on Aks configuration
from azureml.core.webservice import Webservice
from azureml.core.webservice import AksWebservice
aks_config = AksWebservice.deploy_configuration(collect_model_data=True)

In [40]:
# Load AKS compute target
from azureml.core.compute import ComputeTarget
aks_target = ComputeTarget(ws, "dagaldaks")
print(aks_target.cluster_resource_id)

/subscriptions/5f08d643-1910-4a38-a7c7-84a39d4f42e0/resourcegroups/dagald/providers/Microsoft.ContainerService/managedClusters/dagaldaks9db6113849e


In [41]:
%%time
aks_service_name ='mdc-collect-a16-dev3'

aks_service = Webservice.deploy_from_image(workspace = ws, 
                                           name = aks_service_name,
                                           image = image,
                                           deployment_config = aks_config,
                                           deployment_target = aks_target)
aks_service.wait_for_deployment(show_output = True)
print(aks_service.state)



Creating service
Running...............
SucceededAKS service creation operation finished, operation "Succeeded"
Healthy
Wall time: 1min 27s


In [19]:
#from azureml.core.webservice import Webservice
#aks_service_name ='mdc-collect-a16-dev3'
#aks_service = Webservice(ws, aks_service_name)

In [58]:
%%time
import json
import random

test_sample = json.dumps({'data': [
    [random.randint(0,5000),random.randint(0,5000),random.randint(0,5000),random.randint(0,5000),random.randint(0,5000),random.randint(0,5000),random.randint(0,5000),random.randint(0,5000),random.randint(0,5000),random.randint(0,5000)]]})
test_sample = bytes(test_sample,encoding = 'utf8')

prediction = aks_service.run(input_data = test_sample)
print(prediction)

[3103687.164800045]
Wall time: 33.4 ms


In [59]:
import json
import random

for x in range(100):
    print("Iteration: {}".format(x))
    test_sample = json.dumps({'data': [
        [random.randint(0,5000),random.randint(0,5000),random.randint(0,5000),random.randint(0,5000),random.randint(0,5000),random.randint(0,5000),random.randint(0,5000),random.randint(0,5000),random.randint(0,5000),random.randint(0,5000)]]})
    test_sample = bytes(test_sample,encoding = 'utf8')

    prediction = aks_service.run(input_data = test_sample)
    print(prediction)

Iteration: 0
[904296.1684907227]
Iteration: 1
[1987759.2026308852]
Iteration: 2
[-160396.29641111576]
Iteration: 3
[1208578.241481231]
Iteration: 4
[1531809.6557628454]
Iteration: 5
[3820340.2015658026]
Iteration: 6
[3123009.0053255735]
Iteration: 7
[1418655.1926470592]
Iteration: 8
[2429108.8037441703]
Iteration: 9
[557493.6826354049]
Iteration: 10
[1466416.5697393778]
Iteration: 11
[1859141.0668826306]
Iteration: 12
[500663.32447939063]
Iteration: 13
[1845049.8048141468]
Iteration: 14
[3679917.2336773262]
Iteration: 15
[2007425.884227089]
Iteration: 16
[1683585.200472948]
Iteration: 17
[1336375.2320060893]
Iteration: 18
[675032.8464713133]
Iteration: 19
[24221.002753377117]
Iteration: 20
[3146660.5916360132]
Iteration: 21
[3085788.3670571516]
Iteration: 22
[779863.2088975098]
Iteration: 23
[2844997.8943293868]
Iteration: 24
[768750.2291966259]
Iteration: 25
[1553163.2768485015]
Iteration: 26
[2911418.1869538]
Iteration: 27
[2834610.3235863084]
Iteration: 28
[1637507.4532304949]
Itera